In [0]:
WITH t AS (
  SELECT 
  m.sales_subregion_level_3 as BU3
  , m.account_name
  , b.account_executive as ae
  , b.last_solution_architect_engaged as sa
  , m.arr_band
  , b.t3m_annualized
  , m.fiscal_year as fiscal_year
  , replace(m.fiscal_year_quarter, '\'', '') as usage_quarter
  , SUM(m.dbu_dollars) as dbu_dollars
  , SUM(m.dbu_dollar_target) as dbu_dollar_target_bottom_up
  , null as submitted_ae_forecast -- the coliumn 'forecast_ae' only includes closed months
  , SUM(m.uc_dbu_dollars) as dbu_dollars_uc
  , SUM(m.dbsql_dbu_dollars) AS dbu_dollars_sql
  , SUM(m.genai_all_dbu_dollars) AS dbu_dollars_genai
  , SUM(m.serverless_dbu_dollars) AS dbu_dollars_serverless
  , SUM(m.serverless_jobs_dbu_dollars) AS dbu_dollars_serverless_jobs
  , SUM(m.dbsql_serverless_dbu_dollars) AS dbu_dollars_serverless_sql
  , SUM(m.lakeflow_connect_dbu_dollars) AS lakeflow_connect_dbu_dollars
  , SUM(m.lakeflow_pipeline_dbu_dollars) AS lakeflow_pipeline_dbu_dollars
  , SUM(m.serverless_real_time_inference_dbu_dollars) AS dbu_dollars_serverless_realtimeinference
  , SUM(m.serverless_all_purpose_dbu_dollars) AS dbu_dollars_serverless_all_purpose
  , SUM(m.sku_type_automated_dbu_dollars) AS dbu_dollars_sku_type_automated
  , SUM(m.sku_type_interactive_dbu_dollars) AS dbu_dollars_sku_type_interactive
  , SUM(m.azure_dbu_dollars) AS dbu_dollars_azure
  , SUM(m.aws_dbu_dollars) AS dbu_dollars_aws
  , SUM(m.gcp_dbu_dollars) AS dbu_dollars_gcp
  FROM main.gtm_gold.account_consumption_monthly as m
  --FROM main.gtm_data.c360_consumption_account_monthly as m --LEGACY
  LEFT OUTER JOIN main.gtm_silver.account_dim as b
  ON m.account_id = b.account_id
  WHERE m.sales_subregion_level_2 = 'Italy'
  --AND account_name = 'Eni'
  AND fiscal_year IN (2025, 2026) -- only completed quarter from curr_fye fiscal and prev_fyious fiscal
  GROUP BY ALL
)

SELECT

/***** Q3 ANALYSIS *****/
BU3
, account_name
, ae
, sa
, arr_band
, t3m_annualized
, SUM(dbu_dollar_target_bottom_up) as dbu_dollar_target_bottom_up
, SUM(submitted_ae_forecast) as submitted_ae_forecast

, SUM(curr_fy_q3_dbu_dollars) as curr_fy_q3_dbu_dollars
, SUM(curr_fy_q3_dbu_dollars - curr_fy_q2_dbu_dollars) as incr_q3_dbu_dollars
, SUM(try_divide((curr_fy_q3_dbu_dollars - curr_fy_q2_dbu_dollars), curr_fy_q2_dbu_dollars)) as curr_fy_q3_qoq_growth

--dbu_dollars_uc
, SUM(curr_fy_q3_dbu_dollars_uc) as curr_fy_q3_dbu_dollars_uc
, SUM(TRY_DIVIDE(curr_fy_q3_dbu_dollars_uc, curr_fy_q3_dbu_dollars*.8)) as q3_uc_percent_of_dbu_dollars
, SUM(curr_fy_q3_dbu_dollars_uc - curr_fy_q2_dbu_dollars_uc) as incr_q3_dbu_dollars_uc
, SUM(try_divide((curr_fy_q3_dbu_dollars_uc - curr_fy_q2_dbu_dollars_uc), curr_fy_q2_dbu_dollars_uc)) as curr_fy_q3_qoq_growth_uc

--dbu_dollars_sql
, SUM(curr_fy_q3_dbu_dollars_sql) as curr_fy_q3_dbu_dollars_sql
, SUM(TRY_DIVIDE(curr_fy_q3_dbu_dollars_sql, curr_fy_q3_dbu_dollars)) as q3_sql_percent_of_dbu_dollars
, SUM(curr_fy_q3_dbu_dollars_sql - curr_fy_q2_dbu_dollars_sql) as incr_q3_dbu_dollars_sql
, SUM(try_divide((curr_fy_q3_dbu_dollars_sql - curr_fy_q2_dbu_dollars_sql), curr_fy_q2_dbu_dollars_sql)) as curr_fy_q3_qoq_growth_sql

--dbu_dollars_genai
, SUM(curr_fy_q3_dbu_dollars_genai) as curr_fy_q3_dbu_dollars_genai
, SUM(TRY_DIVIDE(curr_fy_q3_dbu_dollars_genai, curr_fy_q3_dbu_dollars)) as q3_genai_percent_of_dbu_dollars
, SUM(curr_fy_q3_dbu_dollars_genai - curr_fy_q2_dbu_dollars_genai) as incr_q3_dbu_dollars_genai
, SUM(try_divide((curr_fy_q3_dbu_dollars_genai - curr_fy_q2_dbu_dollars_genai), curr_fy_q2_dbu_dollars_genai)) as curr_fy_q3_qoq_growth_genai

--dbu_dollars_serverless
, SUM(curr_fy_q3_dbu_dollars_serverless) as curr_fy_q3_dbu_dollars_serverless
, SUM(TRY_DIVIDE(curr_fy_q3_dbu_dollars_serverless, curr_fy_q3_dbu_dollars)) as q3_serverless_percent_of_dbu_dollars
, SUM(curr_fy_q3_dbu_dollars_serverless - curr_fy_q2_dbu_dollars_serverless) as incr_q3_dbu_dollars_serverless
, SUM(try_divide((curr_fy_q3_dbu_dollars_serverless - curr_fy_q2_dbu_dollars_serverless), curr_fy_q2_dbu_dollars_serverless)) as curr_fy_q3_qoq_growth_serverless

--lakeflow_connect_dbu_dollars
, SUM(curr_fy_q3_lakeflow_connect_dbu_dollars) as curr_fy_q3_lakeflow_connect_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_q3_lakeflow_connect_dbu_dollars, curr_fy_q3_dbu_dollars)) as q3_lakeflow_connect_percent_of_dbu_dollars
, SUM(curr_fy_q3_lakeflow_connect_dbu_dollars - curr_fy_q2_lakeflow_connect_dbu_dollars) as incr_q3_lakeflow_connect_dbu_dollars
, SUM(try_divide((curr_fy_q3_lakeflow_connect_dbu_dollars - curr_fy_q2_lakeflow_connect_dbu_dollars), curr_fy_q2_lakeflow_connect_dbu_dollars)) as curr_fy_q3_qoq_growth_lakeflow_connect

--lakeflow_pipeline_dbu_dollars
, SUM(curr_fy_q3_lakeflow_pipeline_dbu_dollars) as curr_fy_q3_lakeflow_pipeline_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_q3_lakeflow_pipeline_dbu_dollars, curr_fy_q3_dbu_dollars)) as q3_lakeflow_pipeline_percent_of_dbu_dollars
, SUM(curr_fy_q3_lakeflow_pipeline_dbu_dollars - curr_fy_q2_lakeflow_pipeline_dbu_dollars) as incr_q3_lakeflow_pipeline_dbu_dollars
, SUM(try_divide((curr_fy_q3_lakeflow_pipeline_dbu_dollars - curr_fy_q2_lakeflow_pipeline_dbu_dollars), curr_fy_q2_lakeflow_pipeline_dbu_dollars)) as curr_fy_q3_qoq_growth_lakeflow_pipeline

/**** H1 ANALYSIS ****/
/*
, SUM(TRY_DIVIDE(curr_fy_q2_dbu_dollars_uc, curr_fy_q2_dbu_dollars*.8)) as q2_uc_percent_of_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_q2_dbu_dollars_sql, curr_fy_q2_dbu_dollars)) as q2_sql_percent_of_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_q2_dbu_dollars_genai, curr_fy_q2_dbu_dollars)) as q2_genai_percent_of_dbu_dollars
, SUM(TRY_DIVIDE(curr_fy_q2_dbu_dollars_serverless, curr_fy_q2_dbu_dollars)) as q2_serverless_percent_of_dbu_dollars
--, SUM(TRY_DIVIDE(curr_fy_q2_dbu_dollars_serverless_sql, curr_fy_q2_dbu_dollars_sql)) as q2_serverless_sql_percent_of_sql_dbu_dollars

, SUM((curr_fy_q2_dbu_dollars - curr_fy_q1_dbu_dollars) + (curr_fy_q1_dbu_dollars - prev_fy_q4_dbu_dollars)) as incr_h1_dbu_dollars
, SUM(try_divide((curr_fy_q1_dbu_dollars - prev_fy_q4_dbu_dollars), prev_fy_q4_dbu_dollars)) as curr_fy_q1_qoq_growth
, SUM(try_divide((curr_fy_q2_dbu_dollars - curr_fy_q1_dbu_dollars), curr_fy_q1_dbu_dollars)) as curr_fy_q2_qoq_growth

--dbu_dollars_uc
, SUM((curr_fy_q2_dbu_dollars_uc - curr_fy_q1_dbu_dollars_uc) + (curr_fy_q1_dbu_dollars_uc - prev_fy_q4_dbu_dollars_uc)) as incr_h1_dbu_dollars_uc
, SUM(try_divide((curr_fy_q1_dbu_dollars_uc - prev_fy_q4_dbu_dollars_uc), prev_fy_q4_dbu_dollars_uc)) as curr_fy_q1_qoq_growth_uc
, SUM(try_divide((curr_fy_q2_dbu_dollars_uc - curr_fy_q1_dbu_dollars_uc), curr_fy_q1_dbu_dollars_uc)) as curr_fy_q2_qoq_growth_uc

--dbu_dollars_sql
, SUM((curr_fy_q2_dbu_dollars_sql - curr_fy_q1_dbu_dollars_sql) + (curr_fy_q1_dbu_dollars_sql - prev_fy_q4_dbu_dollars_sql)) as incr_h1_dbu_dollars_sql
, SUM(try_divide((curr_fy_q1_dbu_dollars_sql - prev_fy_q4_dbu_dollars_sql), prev_fy_q4_dbu_dollars_sql)) as curr_fy_q1_qoq_growth_sql
, SUM(try_divide((curr_fy_q2_dbu_dollars_sql - curr_fy_q1_dbu_dollars_sql), curr_fy_q1_dbu_dollars_sql)) as curr_fy_q2_qoq_growth_sql

--dbu_dollars_genai
, SUM((curr_fy_q2_dbu_dollars_genai - curr_fy_q1_dbu_dollars_genai) + (curr_fy_q1_dbu_dollars_genai - prev_fy_q4_dbu_dollars_genai)) as incr_h1_dbu_dollars_genai
, SUM(try_divide((curr_fy_q1_dbu_dollars_genai - prev_fy_q4_dbu_dollars_genai), prev_fy_q4_dbu_dollars_genai)) as curr_fy_q1_qoq_growth_genai
, SUM(try_divide((curr_fy_q2_dbu_dollars_genai - curr_fy_q1_dbu_dollars_genai), curr_fy_q1_dbu_dollars_genai)) as curr_fy_q2_qoq_growth_genai

--dbu_dollars_serverless
, SUM((curr_fy_q2_dbu_dollars_serverless - curr_fy_q1_dbu_dollars_serverless) + (curr_fy_q1_dbu_dollars_serverless - prev_fy_q4_dbu_dollars_serverless)) as incr_h1_dbu_dollars_serverless
, SUM(try_divide((curr_fy_q1_dbu_dollars_serverless - prev_fy_q4_dbu_dollars_serverless), prev_fy_q4_dbu_dollars_serverless)) as curr_fy_q1_qoq_growth_serverless
, SUM(try_divide((curr_fy_q2_dbu_dollars_serverless - curr_fy_q1_dbu_dollars_serverless), curr_fy_q1_dbu_dollars_serverless)) as curr_fy_q2_qoq_growth_serverless
*/

FROM t  

PIVOT (
  SUM(dbu_dollars) as dbu_dollars
  , SUM(dbu_dollars_uc) as dbu_dollars_uc
  , SUM(dbu_dollars_sql) AS dbu_dollars_sql
  , SUM(dbu_dollars_genai) as dbu_dollars_genai
  , SUM(dbu_dollars_serverless) as dbu_dollars_serverless
  , SUM(dbu_dollars_serverless_jobs) as dbu_dollars_serverless_jobs
  , SUM(dbu_dollars_serverless_sql) as dbu_dollars_serverless_sql
  , SUM(dbu_dollars_serverless_realtimeinference) as dbu_dollars_serverless_realtimeinference
  , SUM(lakeflow_connect_dbu_dollars) as lakeflow_connect_dbu_dollars
  , SUM(lakeflow_pipeline_dbu_dollars) as lakeflow_pipeline_dbu_dollars
  , SUM(dbu_dollars_serverless_all_purpose) as dbu_dollars_serverless_realtimeinference
  , SUM(dbu_dollars_serverless_all_purpose) as dbu_dollars_serverless_all_purpose
  , SUM(dbu_dollars_sku_type_automated) as dbu_dollars_sku_type_automated
  , SUM(dbu_dollars_sku_type_interactive) as dbu_dollars_sku_type_interactive
  , SUM(dbu_dollars_azure) as dbu_dollars_azure
  , SUM(dbu_dollars_aws) as dbu_dollars_aws
  , SUM(dbu_dollars_gcp) as dbu_dollars_gcp
  FOR usage_quarter IN (
    'FY25 Q1' as prev_fy_q1, 'FY25 Q2' as prev_fy_q2, 'FY25 Q3' as prev_fy_q3, 'FY25 Q4' as prev_fy_q4, 
    'FY26 Q1' as curr_fy_q1, 'FY26 Q2' as curr_fy_q2, 'FY26 Q3' as curr_fy_q3, 'FY26 Q4' as curr_fy_q4)
)
GROUP BY ALL